# **Prediksi Gagal Bayar Kartu Kredit Menggunakan Machine Learning**
## UTS Prediksi Modern & Machine Learning


Analisis ini bertujuan memprediksi apakah nasabah kartu kredit akan mengalami gagal bayar pada bulan berikutnya. Eksperimen dilakukan secara bertahap untuk mengevaluasi pengaruh persiapan data, scaling, pemilihan model, hyperparameter tuning, dan penggunaan metode advanced.
F1-score untuk kelas gagal bayar digunakan sebagai metrik utama, sedangkan accuracy, precision, dan recall menjadi metrik pendukung. Evaluasi menggunakan 5-fold cross validation pada training set serta pengujian pada test set.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Experiment 0 — Data Understanding & Preparation

Eksperimen ini bertujuan memahami struktur dan kondisi dataset sebelum pemodelan. Pemeriksaan mencakup ukuran data, tipe variabel, missing value, duplikat, distribusi target, kategori yang perlu ditinjau, serta rentang nilai predictor.

In [2]:
df = pd.read_csv('/content/UCI_Credit_Card.csv')
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


Memeriksa Dimensi, Struktur, dan Tipe Data

In [3]:
print('Ukuran data:', df.shape)
df.info()

Ukuran data: (30000, 25)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-null  float64
 13  BILL_A

Memeriksa Missing Value

In [4]:
df.isna().sum()

,0
ID,0
LIMIT_BAL,0
SEX,0
EDUCATION,0
MARRIAGE,0
AGE,0
PAY_0,0
PAY_2,0
PAY_3,0
PAY_4,0


Memeriksa Duplikat

In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.drop(columns="ID").duplicated().sum()

np.int64(35)

Mengidentifikasi Target dan Distribusi Kelas

In [7]:
df["default.payment.next.month"].value_counts()

,count
default.payment.next.month,
0,23364
1,6636


In [8]:
df["default.payment.next.month"].value_counts(normalize=True) * 100

,proportion
default.payment.next.month,
0,77.88
1,22.12


Memeriksa Kode Kategori

In [9]:
for kolom in ["SEX", "EDUCATION", "MARRIAGE"]:
    print(kolom)
    print(df[kolom].value_counts().sort_index())
    print()

SEX
SEX
1    11888
2    18112
Name: count, dtype: int64

EDUCATION
EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

MARRIAGE
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64



Memeriksa Statistik Deskriptif dan Nilai Tidak Wajar

In [10]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,30000.0,15000.500000,8660.398374,1.0,7500.75,15000.5,22500.25,30000.0
LIMIT_BAL,30000.0,167484.322667,129747.661567,10000.0,50000.00,140000.0,240000.00,1000000.0
SEX,30000.0,1.603733,0.489129,1.0,1.00,2.0,2.00,2.0
EDUCATION,30000.0,1.853133,0.790349,0.0,1.00,2.0,2.00,6.0
MARRIAGE,30000.0,1.551867,0.521970,0.0,1.00,2.0,2.00,3.0
AGE,30000.0,35.485500,9.217904,21.0,28.00,34.0,41.00,79.0
PAY_0,30000.0,-0.016700,1.123802,-2.0,-1.00,0.0,0.00,8.0
PAY_2,30000.0,-0.133767,1.197186,-2.0,-1.00,0.0,0.00,8.0
PAY_3,30000.0,-0.166200,1.196868,-2.0,-1.00,0.0,0.00,8.0
PAY_4,30000.0,-0.220667,1.169139,-2.0,-1.00,0.0,0.00,8.0


Menyiapkan Salinan dan Mengelompokkan Kategori

In [11]:
data = df.copy()

In [12]:
data["EDUCATION"] = data["EDUCATION"].replace({
    5: 0,
    6: 0
})

In [13]:
for kolom in ["SEX", "EDUCATION", "MARRIAGE"]:
    print(kolom)
    print(data[kolom].value_counts().sort_index())
    print()

SEX
SEX
1    11888
2    18112
Name: count, dtype: int64

EDUCATION
EDUCATION
0      345
1    10585
2    14030
3     4917
4      123
Name: count, dtype: int64

MARRIAGE
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64



Menentukan Predictor dan Target

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
X = data.drop(columns=["ID", "default.payment.next.month"])
y = data["default.payment.next.month"]

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
pola_sama = X_train.drop_duplicates().merge(
    X_test.drop_duplicates(),
    on=list(X.columns),
    how="inner"
)

print("Jumlah pola predictor yang sama:", len(pola_sama))

Jumlah pola predictor yang sama: 15


Membentuk Kelompok Predictor Identik

Observasi dikelompokkan berdasarkan kesamaan seluruh 23 predictor. ID dan target tidak digunakan untuk membentuk kelompok.

In [21]:
groups = X.groupby(
    list(X.columns),
    dropna=False
).ngroup()

print("Jumlah observasi:", len(X))
print("Jumlah kelompok :", groups.nunique())

Jumlah observasi: 30000
Jumlah kelompok : 29944


Membagi Data Berdasarkan Kelompok

In [22]:
from sklearn.model_selection import GroupShuffleSplit

split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    split.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

In [23]:
print("Training:", X_train.shape)
print("Test    :", X_test.shape)

print("\nPersentase target training:")
print(y_train.value_counts(normalize=True).sort_index() * 100)

print("\nPersentase target test:")
print(y_test.value_counts(normalize=True).sort_index() * 100)

kelompok_sama = set(groups_train) & set(groups_test)

print("\nKelompok yang ada di training dan test:", len(kelompok_sama))

Training: (24000, 23)
Test    : (6000, 23)

Persentase target training:
default.payment.next.month
0    78.004167
1    21.995833
Name: proportion, dtype: float64

Persentase target test:
default.payment.next.month
0    77.383333
1    22.616667
Name: proportion, dtype: float64

Kelompok yang ada di training dan test: 0


# Experiment 1 — Baseline Model

Logistic Regression dibangun sebagai model awal tanpa scaling dan encoding tambahan. Hasil baseline menjadi pembanding untuk menilai perubahan performa pada eksperimen berikutnya.


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

Membentuk Logistic Regression Baseline

In [25]:
baseline = LogisticRegression(
    solver="newton-cholesky",
    max_iter=1000,
    random_state=42
)

Menyiapkan 5-Fold Cross Validation Berdasarkan Kelompok

In [26]:
from sklearn.model_selection import StratifiedGroupKFold

cv_kelompok = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv = list(
    cv_kelompok.split(
        X_train,
        y_train,
        groups=groups_train
    )
)

Evaluasi Cross Validation Baseline

In [27]:
cv_f1 = cross_val_score(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring="f1"
)

print("F1-score setiap fold:", cv_f1)
print("Rata-rata CV F1-score:", cv_f1.mean())

F1-score setiap fold: [0.34067548 0.34090909 0.32968636 0.37689133 0.34882059]
Rata-rata CV F1-score: 0.3473965697705349


Pelatihan Akhir dan Prediksi Test Set

In [28]:
baseline.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='newton-cholesky')

In [29]:
baseline.fit(X_train, y_train)

y_pred_baseline = baseline.predict(X_test)

Evaluasi Baseline pada Test Set

In [30]:
hasil_exp1 = pd.DataFrame({
    "Model": ["Logistic Regression Baseline"],
    "CV F1": [cv_f1.mean()],
    "Test Accuracy": [accuracy_score(y_test, y_pred_baseline)],
    "Test Precision": [
        precision_score(y_test, y_pred_baseline, zero_division=0)
    ],
    "Test Recall": [
        recall_score(y_test, y_pred_baseline, zero_division=0)
    ],
    "Test F1": [
        f1_score(y_test, y_pred_baseline, zero_division=0)
    ]
})

hasil_exp1.round(4)

,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,Logistic Regression Baseline,0.3474,0.8095,0.75,0.2366,0.3597


Baseline memperoleh accuracy 0,8095, precision 0,7500, recall 0,2366, dan Test F1 0,3597. Sebanyak 75% prediksi gagal bayar benar, tetapi model hanya mendeteksi sekitar 23,66% nasabah yang benar-benar gagal bayar. Kemampuan deteksi kelas positif masih terbatas.

# Experiment 2 — Preprocessing

Eksperimen ini menguji pengaruh StandardScaler terhadap Logistic Regression. Model menggunakan pengaturan yang sama dengan baseline dan tanpa encoding tambahan, sehingga fokus perbandingan adalah penambahan scaling

In [89]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

Membentuk Pipeline Scaling dan Model

In [92]:
model_exp2 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=42
    ))
])

Evaluasi Cross Validation

In [93]:
cv_f1_exp2 = cross_val_score(
    model_exp2,
    X_train,
    y_train,
    cv=cv,
    scoring="f1"
)

print("F1-score setiap fold:", cv_f1_exp2)
print("Rata-rata CV F1-score:", round(cv_f1_exp2.mean(), 4))

F1-score setiap fold: [0.34092579 0.34115139 0.32968636 0.37800687 0.34763948]
Rata-rata CV F1-score: 0.3475


Pelatihan dan Evaluasi Test Set

In [94]:
model_exp2.fit(X_train, y_train)

y_pred_exp2 = model_exp2.predict(X_test)

In [95]:
hasil_exp2 = pd.DataFrame({
    "Experiment": [2],
    "Model": ["Logistic Regression + Preprocessing"],
    "CV F1": [cv_f1_exp2.mean()],
    "Test Accuracy": [accuracy_score(y_test, y_pred_exp2)],
    "Test Precision": [
        precision_score(y_test, y_pred_exp2, zero_division=0)
    ],
    "Test Recall": [
        recall_score(y_test, y_pred_exp2, zero_division=0)
    ],
    "Test F1": [
        f1_score(y_test, y_pred_exp2, zero_division=0)
    ]
})

hasil_exp2.round(4)

,Experiment,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,2,Logistic Regression + Preprocessing,0.3475,0.8095,0.75,0.2366,0.3597


Perbandingan dengan Baseline

In [44]:
perbandingan_exp1_exp2 = pd.concat(
    [hasil_exp1, hasil_exp2],
    ignore_index=True
)

perbandingan_exp1_exp2.round(4)

,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1,Experiment
0,Logistic Regression Baseline,0.3474,0.8095,0.75,0.2366,0.3597,NaN
1,Logistic Regression + Preprocessing,0.3475,0.8095,0.75,0.2366,0.3597,2.0


Konfigurasi dengan scaling belum menunjukkan peningkatan performa yang berarti. CV F1 berubah sangat kecil dari 0,3474 menjadi 0,3475, sementara seluruh metrik test tetap sama hingga empat desimal.

# Experiment 3 — Model Comparison

Eksperimen ini membandingkan Logistic Regression, Decision Tree, dan Random Forest menggunakan training-test set serta pembagian cross validation yang sama. Logistic Regression menggunakan scaling melalui Pipeline, sedangkan model pohon menggunakan predictor tanpa scaling.

In [45]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [46]:
model_exp3 = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            solver="lbfgs",
            max_iter=1000,
            random_state=42
        ))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

In [47]:
hasil = []

for nama, model in model_exp3.items():
    print("Memproses:", nama)

    # Cross validation pada training set
    skor_cv = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1"
    )

    # Training dan prediksi test set
    model.fit(X_train, y_train)
    prediksi = model.predict(X_test)

    # Simpan hasil evaluasi
    hasil.append({
        "Experiment": 3,
        "Model": nama,
        "CV F1": skor_cv.mean(),
        "Test Accuracy": accuracy_score(y_test, prediksi),
        "Test Precision": precision_score(
            y_test, prediksi, zero_division=0
        ),
        "Test Recall": recall_score(
            y_test, prediksi, zero_division=0
        ),
        "Test F1": f1_score(
            y_test, prediksi, zero_division=0
        )
    })

Memproses: Logistic Regression
Memproses: Decision Tree
Memproses: Random Forest


In [48]:
hasil_exp3 = pd.DataFrame(hasil)

hasil_exp3 = hasil_exp3.sort_values(
    by="CV F1",
    ascending=False
).reset_index(drop=True)

hasil_exp3.round(4)

,Experiment,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,3,Random Forest,0.4621,0.8185,0.6831,0.3685,0.4787
1,3,Decision Tree,0.3996,0.7315,0.4085,0.4178,0.4131
2,3,Logistic Regression,0.3475,0.8095,0.7500,0.2366,0.3597


Random Forest menghasilkan CV F1 tertinggi sebesar 0,4621 dan Test F1 tertinggi sebesar 0,4787. Decision Tree memperoleh CV F1 0,3996 dan Test F1 0,4131, sedangkan Logistic Regression memperoleh 0,3475 dan 0,3597. Decision Tree memiliki recall tertinggi sebesar 0,4178, tetapi precision lebih rendah.

In [49]:
nama_model_terbaik = hasil_exp3.loc[0, "Model"]
model_terbaik = model_exp3[nama_model_terbaik]

print("Model terbaik:", nama_model_terbaik)
print("CV F1:", round(hasil_exp3.loc[0, "CV F1"], 4))

Model terbaik: Random Forest
CV F1: 0.4621


Random Forest dipilih untuk Experiment 4 berdasarkan CV F1 tertinggi, didukung Test F1 tertinggi. Model ini memberikan keseimbangan precision dan recall terbaik di antara tiga konfigurasi yang diuji.

# Experiment 4 — Hyperparameter Tuning

Hyperparameter tuning dilakukan pada Random Forest sebagai model terbaik Experiment 3. GridSearchCV menggunakan 5-fold cross validation dan F1-score untuk memilih kombinasi parameter terbaik hanya berdasarkan training set.

In [50]:
from sklearn.model_selection import GridSearchCV

Menentukan Parameter yang Diuji

In [51]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, None],
    "min_samples_leaf": [1, 5],
    "class_weight": [None, "balanced"]
}

print(param_grid)

{'n_estimators': [100, 200], 'max_depth': [10, None], 'min_samples_leaf': [1, 5], 'class_weight': [None, 'balanced']}


In [52]:
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ),
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True
)

In [53]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


GridSearchCV(cv=[(array([    0,     1,     4, ..., 23996, 23998, 23999]),
                  array([    2,     3,    16, ..., 23988, 23993, 23997])),
                 (array([    1,     2,     3, ..., 23994, 23997, 23999]),
                  array([    0,     4,     7, ..., 23995, 23996, 23998])),
                 (array([    0,     1,     2, ..., 23997, 23998, 23999]),
                  array([   15,    17,    26, ..., 23982, 23989, 23992])),
                 (array([    0,     2,     3, ..., 23996, 23997, 23998]),
                  array([    1,     5,    10, ..., 23991, 23994, 23999])),
                 (array([    0,     1,     2, ..., 23997, 23998, 23999]),
                  array([    6,     8,     9, ..., 23967, 23976, 23981]))],
             estimator=RandomForestClassifier(n_jobs=1, random_state=42),
             n_jobs=-1,
             param_grid={'class_weight': [None, 'balanced'],
                         'max_depth': [10, None], 'min_samples_leaf': [1, 5],
                         'n_estimators': [100, 200]},
             scoring='f1', verbose=1)

In [54]:
print("Parameter terbaik:", grid_search.best_params_)
print("Best CV F1-score:", round(grid_search.best_score_, 4))

Parameter terbaik: {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 100}
Best CV F1-score: 0.5384


Parameter terbaik adalah n_estimators=100, max_depth=10, min_samples_leaf=5, dan class_weight='balanced', dengan best CV F1 0,5384. Pembobotan kelas digunakan untuk memberikan bobot lebih besar kepada kelas yang jumlahnya lebih sedikit.

In [55]:
hasil_grid = pd.DataFrame(grid_search.cv_results_)

hasil_grid[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score").reset_index(drop=True)

,params,mean_test_score,std_test_score,rank_test_score
0,"{'class_weight': 'balanced', 'max_depth': 10, ...",0.538408,0.008318,1
1,"{'class_weight': 'balanced', 'max_depth': 10, ...",0.538246,0.008195,2
2,"{'class_weight': 'balanced', 'max_depth': 10, ...",0.536160,0.010118,3
3,"{'class_weight': 'balanced', 'max_depth': 10, ...",0.534293,0.009597,4
4,"{'class_weight': 'balanced', 'max_depth': None...",0.531914,0.009582,5
5,"{'class_weight': 'balanced', 'max_depth': None...",0.530102,0.005936,6
6,"{'class_weight': None, 'max_depth': 10, 'min_s...",0.467486,0.007954,7
7,"{'class_weight': None, 'max_depth': 10, 'min_s...",0.467280,0.009555,8
8,"{'class_weight': None, 'max_depth': None, 'min...",0.466364,0.009782,9
9,"{'class_weight': None, 'max_depth': None, 'min...",0.465907,0.009684,10


In [56]:
model_tuned = grid_search.best_estimator_

y_pred_exp4 = model_tuned.predict(X_test)

In [57]:
hasil_exp4 = pd.DataFrame({
    "Experiment": [4],
    "Model": ["Random Forest + Tuning"],
    "CV F1": [grid_search.best_score_],
    "Test Accuracy": [accuracy_score(y_test, y_pred_exp4)],
    "Test Precision": [
        precision_score(y_test, y_pred_exp4, zero_division=0)
    ],
    "Test Recall": [
        recall_score(y_test, y_pred_exp4, zero_division=0)
    ],
    "Test F1": [
        f1_score(y_test, y_pred_exp4, zero_division=0)
    ]
})

hasil_exp4.round(4)

,Experiment,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,4,Random Forest + Tuning,0.5384,0.7958,0.5456,0.5814,0.563


In [58]:
hasil_rf_sebelum = hasil_exp3[
    hasil_exp3["Model"] == "Random Forest"
]

perbandingan_tuning = pd.concat(
    [hasil_rf_sebelum, hasil_exp4],
    ignore_index=True
)

perbandingan_tuning.round(4)

,Experiment,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,3,Random Forest,0.4621,0.8185,0.6831,0.3685,0.4787
1,4,Random Forest + Tuning,0.5384,0.7958,0.5456,0.5814,0.5630


Tuning meningkatkan CV F1 dari 0,4621 menjadi 0,5384, Test F1 dari 0,4787 menjadi 0,5630, dan recall dari 0,3685 menjadi 0,5814. Precision serta accuracy menurun, menunjukkan peningkatan deteksi gagal bayar disertai lebih banyak prediksi positif yang keliru. Berdasarkan F1-score, tuning meningkatkan performa.

# Experiment 5 — Advanced Model

Gradient Boosting digunakan sebagai metode advanced yang membangun pohon secara bertahap untuk memperbaiki kesalahan prediksi. Model dievaluasi menggunakan prosedur yang sama dan dibandingkan dengan Random Forest hasil tuning.

In [73]:
from sklearn.ensemble import GradientBoostingClassifier

In [74]:
model_exp5 = GradientBoostingClassifier(
    random_state=42
)

In [75]:
cv_f1_exp5 = cross_val_score(
    model_exp5,
    X_train,
    y_train,
    cv=cv,
    scoring="f1"
)

print("F1-score setiap fold:", cv_f1_exp5)
print("Rata-rata CV F1-score:", round(cv_f1_exp5.mean(), 4))

F1-score setiap fold: [0.47243108 0.4648582  0.46779237 0.47852029 0.4749082 ]
Rata-rata CV F1-score: 0.4717


In [76]:
model_exp5.fit(X_train, y_train)

y_pred_exp5 = model_exp5.predict(X_test)

In [77]:
hasil_exp5 = pd.DataFrame({
    "Experiment": [5],
    "Model": ["Gradient Boosting"],
    "CV F1": [cv_f1_exp5.mean()],
    "Test Accuracy": [accuracy_score(y_test, y_pred_exp5)],
    "Test Precision": [
        precision_score(y_test, y_pred_exp5, zero_division=0)
    ],
    "Test Recall": [
        recall_score(y_test, y_pred_exp5, zero_division=0)
    ],
    "Test F1": [
        f1_score(y_test, y_pred_exp5, zero_division=0)
    ]
})

hasil_exp5.round(4)

,Experiment,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,5,Gradient Boosting,0.4717,0.822,0.7027,0.3692,0.4841


In [78]:
perbandingan_exp4_exp5 = pd.concat(
    [hasil_exp4, hasil_exp5],
    ignore_index=True
)

perbandingan_exp4_exp5.round(4)

,Experiment,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,4,Random Forest + Tuning,0.5384,0.7958,0.5456,0.5814,0.5630
1,5,Gradient Boosting,0.4717,0.8220,0.7027,0.3692,0.4841


Gradient Boosting belum mengungguli Random Forest hasil tuning berdasarkan CV F1 dan Test F1. Meskipun accuracy dan precision lebih tinggi, recall lebih rendah. Metode advanced yang diuji tidak memberikan peningkatan berdasarkan metrik utama.

Summary of Experiment Results

In [79]:
hasil_exp1["Experiment"] = 1
hasil_exp2["Model"] = "Logistic Regression + Scaling"

ringkasan = pd.concat(
    [hasil_exp1, hasil_exp2, hasil_exp3, hasil_exp4, hasil_exp5],
    ignore_index=True
)

ringkasan[
    ["Experiment", "Model", "CV F1", "Test Accuracy",
     "Test Precision", "Test Recall", "Test F1"]
].round(4)

,Experiment,Model,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1
0,1,Logistic Regression Baseline,0.3474,0.8095,0.7500,0.2366,0.3597
1,2,Logistic Regression + Scaling,0.3475,0.8095,0.7500,0.2366,0.3597
2,3,Random Forest,0.4621,0.8185,0.6831,0.3685,0.4787
3,3,Decision Tree,0.3996,0.7315,0.4085,0.4178,0.4131
4,3,Logistic Regression,0.3475,0.8095,0.7500,0.2366,0.3597
5,4,Random Forest + Tuning,0.5384,0.7958,0.5456,0.5814,0.5630
6,5,Gradient Boosting,0.4717,0.8220,0.7027,0.3692,0.4841


Random Forest hasil tuning memperoleh CV F1 tertinggi sebesar 0,5384 dan Test F1 tertinggi sebesar 0,5630. Gradient Boosting memperoleh accuracy tertinggi sebesar 0,8220, tetapi F1 lebih rendah. Hasil ini menunjukkan bahwa accuracy tertinggi tidak selalu menghasilkan kemampuan deteksi gagal bayar terbaik.

## Kesimpulan

Random Forest hasil tuning memberikan performa terbaik berdasarkan F1-score di antara konfigurasi yang diuji. Model final menggunakan 100 pohon, kedalaman maksimum 10, minimum 5 observasi pada daun, dan pembobotan kelas balanced. Model memperoleh CV F1 0,5384, Test F1 0,5630, precision 0,5456, dan recall 0,5814. Kemampuan deteksi gagal bayar meningkat setelah tuning, tetapi masih terdapat kasus gagal bayar yang tidak terdeteksi serta prediksi positif yang keliru.